<a href="https://colab.research.google.com/github/iamharshnegi/LLM-LoRA-FineTuning/blob/main/LLM_LoRA_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets peft accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00


In [7]:
!pip install --upgrade torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 14.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np

# Verify PyTorch is using the GPU
print(f"Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

Using device: cuda


In [12]:
# =====================================================================
# 2. LOAD DATASET & BASE MODEL
# =====================================================================
# We use a small subset of the IMDB movie review dataset for speed
dataset = load_dataset("stanfordnlp/imdb")

# Load tokenizer and base model (DistilBERT is fast and lightweight)
model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Initialize model for 2-class classification (Positive/Negative)
base_model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# =====================================================================
# 3. PREPROCESS DATA (Tokenization)
# =====================================================================
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

# Tokenize and sample 1,000 rows for training to finish in ~3 minutes
tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =====================================================================
# 4. CONFIGURING PEFT / LoRA (The Fine-Tuning Magic)
# =====================================================================
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence Classification task
    r=8,                        # Rank: size of the adapter matrices
    lora_alpha=16,              # Scaling factor
    lora_dropout=0.1,           # Dropout probability for LoRA layers
    target_modules=["q_lin", "v_lin"] # Target query and value projections in DistilBERT
)

# Wrap the base model with LoRA configurations
model = get_peft_model(base_model, peft_config)

# Print trainable parameters to prove we are freezing the base weights!
model.print_trainable_parameters()

# =====================================================================
# 5. DEFINE TRAINING ARGUMENTS & METRICS
# =====================================================================
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./lora-fine-tuned-sentiment",
    learning_rate=2e-4,              # LoRA typically uses slightly higher learning rates
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,              # 1 epoch is enough to demonstrate the pipeline
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="none"                 # Prevents prompt loops asking for WandB accounts
)

# =====================================================================
# 6. INITIALIZE TRAINER AND START FINE-TUNING
# =====================================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n--- Starting Fine-Tuning ---")
trainer.train()
print("--- Fine-Tuning Complete! ---")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

--- Starting Fine-Tuning ---


Epoch,Training Loss,Validation Loss,Accuracy
1,0.587161,0.569586,0.814000


--- Fine-Tuning Complete! ---


In [13]:
# Save the LoRA adapter weights and tokenizer configuration
model.save_pretrained("./lora-fine-tuned-sentiment")
tokenizer.save_pretrained("./lora-fine-tuned-sentiment")
print("Model saved successfully!")


Model saved successfully!
